# LeetCode 1550 — Three Consecutive Odds bằng Python

**Link đề:** https://leetcode.com/problems/three-consecutive-odds/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def threeConsecutiveOdds(self, arr: list[int]) -> bool:
        ...
```

### Mục tiêu học
1. Kiểm tra số **lẻ** bằng `x % 2 == 1` (và mẹo bit `x & 1`).
2. Hiểu khái niệm **liên tiếp** (consecutive) = các vị trí kề nhau `i, i+1, i+2` — không phải giá trị liên tiếp.
3. Hai kỹ thuật cơ bản trên mảng: **cửa sổ trượt cố định** (window size 3) và **bộ đếm chuỗi liên tiếp** (streak counter).
4. Dùng `range(len(arr) - 2)` đúng biên, tránh `IndexError`; biết `any`, `zip`, slicing.
5. Trace, chứng minh, Time/Space, test đối chiếu, tổng quát hóa thành "k số lẻ liên tiếp".

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — có ba số lẻ đứng liền nhau không?

Cho mảng số nguyên `arr`. Trả về `True` nếu trong mảng có **ba số lẻ liên tiếp** (ba phần tử ở ba vị trí kề nhau, cả ba đều lẻ), ngược lại `False`.

| Ví dụ | `arr` | Output | Giải thích |
|---|---|---|---|
| 1 | `[2,6,4,1]` | `False` | chỉ có một số lẻ |
| 2 | `[1,2,34,3,4,5,7,23,12]` | `True` | `[5,7,23]` ở vị trí 5,6,7 |

**Constraints:** `1 <= arr.length <= 1000`, `1 <= arr[i] <= 1000`.

Lưu ý:
- "Liên tiếp" là **vị trí** liên tiếp: `[1, 3, 5]` ở vị trí 0,1,2 → `True`; còn `[1, 2, 3, 4, 5]` có 3 số lẻ nhưng **không đứng liền** → `False`.
- Mảng có thể ngắn hơn 3 → chắc chắn `False`.

In [1]:
arr = [1, 2, 34, 3, 4, 5, 7, 23, 12]
for i, x in enumerate(arr):
    print(f"vị trí {i}: {x:>2} → {'lẻ' if x % 2 == 1 else 'chẵn'}")

vị trí 0:  1 → lẻ
vị trí 1:  2 → chẵn
vị trí 2: 34 → chẵn
vị trí 3:  3 → lẻ
vị trí 4:  4 → chẵn
vị trí 5:  5 → lẻ
vị trí 6:  7 → lẻ
vị trí 7: 23 → lẻ
vị trí 8: 12 → chẵn


Biểu thức `A if điều_kiện else B` là **toán tử điều kiện** (ternary): cho `A` nếu điều kiện đúng, ngược lại `B`.

## 2. Nền tảng Python cần dùng

### 2.1. Chẵn/lẻ với `%`

`x % 2` là **phần dư** khi chia `x` cho 2: bằng `0` nếu chẵn, `1` nếu lẻ (với `x >= 0`).

In [2]:
for x in [0, 1, 2, 7, 10, 23]:
    print(x, "% 2 =", x % 2, "→ lẻ?", x % 2 == 1)

0 % 2 = 0 → lẻ? False
1 % 2 = 1 → lẻ? True
2 % 2 = 0 → lẻ? False
7 % 2 = 1 → lẻ? True
10 % 2 = 0 → lẻ? False
23 % 2 = 1 → lẻ? True


Với số âm, Python cho `-3 % 2 == 1` (phần dư luôn cùng dấu với số chia), nên `x % 2 == 1` vẫn đúng cho số lẻ âm. Trong C/Java thì `-3 % 2 == -1` — an toàn hơn là viết `x % 2 != 0`. Đề này chỉ có số dương nên không lo.

In [3]:
print(-3 % 2, -4 % 2, (-3) % 2 != 0)

1 0 True


### 2.2. Mẹo bit: `x & 1`

Trong hệ nhị phân, bit cuối của số lẻ là `1`. Phép AND từng bit `x & 1` giữ lại đúng bit cuối → `1` nếu lẻ, `0` nếu chẵn. Nhanh và gặp nhiều trong code thi đấu.

In [4]:
for x in [5, 6, 7, 23]:
    print(f"{x:>2} = {x:05b} (nhị phân) → x & 1 = {x & 1}")

 5 = 00101 (nhị phân) → x & 1 = 1
 6 = 00110 (nhị phân) → x & 1 = 0
 7 = 00111 (nhị phân) → x & 1 = 1
23 = 10111 (nhị phân) → x & 1 = 1


`{x:05b}` in `x` ở dạng nhị phân (`b`), đủ 5 chữ số, thêm `0` ở đầu.

### 2.3. Ba phần tử kề nhau và biên của `range`

Bộ ba bắt đầu ở `i` là `arr[i], arr[i+1], arr[i+2]`. Chỉ số lớn nhất `i + 2` phải `<= len(arr) - 1` → `i <= len(arr) - 3` → dùng `range(len(arr) - 2)`.

In [5]:
arr = [10, 20, 30, 40, 50]
for i in range(len(arr) - 2):
    print(i, arr[i:i + 3])
print("range(len([1,2]) - 2) rỗng:", list(range(len([1, 2]) - 2)))

0 [10, 20, 30]
1 [20, 30, 40]
2 [30, 40, 50]
range(len([1,2]) - 2) rỗng: []


Hai điều cần nhớ:
- `arr[i:i+3]` là **slicing**: lấy từ `i` đến trước `i+3` — đúng 3 phần tử.
- Khi `len(arr) < 3`, `range(âm hoặc 0)` **rỗng**, vòng lặp không chạy → tự nhiên trả `False`, không cần xử lý riêng.

### 2.4. `any()` và `all()`

- `all(iterable)`: `True` nếu **mọi** phần tử đúng.
- `any(iterable)`: `True` nếu **có ít nhất một** phần tử đúng. Cả hai **dừng sớm** khi đã biết kết quả.

In [6]:
print(all(x % 2 == 1 for x in [5, 7, 23]))
print(all(x % 2 == 1 for x in [5, 8, 23]))
print(any(x > 20 for x in [5, 7, 23]))
print("all([]) =", all([]), "| any([]) =", any([]))

True
False
True
all([]) = True | any([]) = False


## 3. Cách 1 — Cửa sổ cố định: thử mọi bộ ba kề nhau

Ý tưởng tự nhiên (và là gợi ý của đề): với mọi `i`, kiểm tra cả 3 số `arr[i], arr[i+1], arr[i+2]` có lẻ không.

In [7]:
def three_odds_window(arr: list[int]) -> bool:
    for i in range(len(arr) - 2):
        if arr[i] % 2 == 1 and arr[i + 1] % 2 == 1 and arr[i + 2] % 2 == 1:
            return True               # tìm thấy → trả ngay
    return False                      # duyệt hết mà không có

print(three_odds_window([2, 6, 4, 1]))
print(three_odds_window([1, 2, 34, 3, 4, 5, 7, 23, 12]))

False
True


Cách viết gọn bằng `any` + `all` (cùng ý tưởng):

In [8]:
def three_odds_any(arr: list[int]) -> bool:
    return any(all(x % 2 == 1 for x in arr[i:i + 3]) for i in range(len(arr) - 2))

print(three_odds_any([2, 6, 4, 1]), three_odds_any([1, 2, 34, 3, 4, 5, 7, 23, 12]))

False True


### 3.1. Có gì "thừa" ở cách cửa sổ?

Mỗi phần tử bị kiểm tra chẵn/lẻ **tới 3 lần** (nó thuộc 3 cửa sổ). Với `k = 3` thì không sao, nhưng nếu đề đổi thành "`k` số lẻ liên tiếp" thì cách này tốn `O(n·k)`. Đếm số phép kiểm tra trên mảng xấu: lặp lại khối `k-1` số lẻ rồi một số chẵn (không bao giờ đủ `k`, và mỗi cửa sổ phải đọc nhiều phần tử mới gặp số chẵn):

In [9]:
def count_checks_window(arr, k):
    checks = 0
    for i in range(len(arr) - k + 1):
        for j in range(i, i + k):
            checks += 1
            if arr[j] % 2 == 0:
                break
        else:
            return checks
    return checks

for k in [3, 10, 50]:
    arr_k = ([1] * (k - 1) + [2]) * (900 // k)
    print(f"k={k:>2}: số phép kiểm tra = {count_checks_window(arr_k, k)} (n = {len(arr_k)})")

k= 3: số phép kiểm tra = 1797 (n = 900)
k=10: số phép kiểm tra = 4905 (n = 900)
k=50: số phép kiểm tra = 21725 (n = 900)


Cú pháp `for ... else`: khối `else` chạy khi vòng `for` **kết thúc tự nhiên** (không bị `break`). Ở đây: duyệt đủ `k` phần tử lẻ mà không `break` → tìm thấy.

## 4. Insight — chỉ cần nhớ "đang có bao nhiêu số lẻ liền nhau"

Đọc từ trái sang phải, giữ biến `streak` = số lượng số lẻ **liên tiếp kết thúc tại vị trí hiện tại**:
- Gặp số lẻ → `streak += 1`.
- Gặp số chẵn → chuỗi bị cắt → `streak = 0`.
- Khi `streak == 3` → trả `True`.

Mỗi phần tử chỉ được nhìn **một lần**, và công thức không phụ thuộc `k`.

In [10]:
streak = 0
for x in [5, 7, 2, 9, 11, 13]:
    streak = streak + 1 if x % 2 == 1 else 0
    print(f"x={x:>2} → streak={streak}")

x= 5 → streak=1
x= 7 → streak=2
x= 2 → streak=0
x= 9 → streak=1
x=11 → streak=2
x=13 → streak=3


## 5. Lời giải chính — Bộ đếm chuỗi liên tiếp (streak)

### 5.1. Từng đoạn
1. `streak = 0`.
2. Với mỗi `x` trong `arr`:
   - `if x % 2 == 1:` → `streak += 1`; nếu `streak == 3` → `return True`.
   - `else:` → `streak = 0`.
3. Hết mảng → `return False`.

### 5.2. Ghép lại

In [11]:
def three_odds_streak(arr: list[int]) -> bool:
    streak = 0
    for x in arr:
        if x % 2 == 1:
            streak += 1
            if streak == 3:
                return True
        else:
            streak = 0
    return False

print(three_odds_streak([2, 6, 4, 1]))
print(three_odds_streak([1, 2, 34, 3, 4, 5, 7, 23, 12]))
print(three_odds_streak([1, 3, 5]))

False
True
True


## 6. TRACE từng bước

Ví dụ 2: `arr = [1,2,34,3,4,5,7,23,12]`

| i | x | lẻ? | streak sau | Quyết định |
|---|---|---|---|---|
| 0 | 1 | ✓ | 1 | tiếp |
| 1 | 2 | ✗ | 0 | reset |
| 2 | 34 | ✗ | 0 | reset |
| 3 | 3 | ✓ | 1 | tiếp |
| 4 | 4 | ✗ | 0 | reset |
| 5 | 5 | ✓ | 1 | tiếp |
| 6 | 7 | ✓ | 2 | tiếp |
| 7 | 23 | ✓ | 3 | **return True** |

Phần tử `12` (vị trí 8) **không bao giờ được đọc** — nhờ return sớm.

In [12]:
def trace_streak(arr):
    streak = 0
    print("arr =", arr)
    for i, x in enumerate(arr):
        odd = x % 2 == 1
        streak = streak + 1 if odd else 0
        note = "RESET" if not odd else ("→ ĐỦ 3, return True" if streak == 3 else "")
        print(f"i={i}, x={x:>2}, lẻ={odd!s:<5}, streak={streak} {note}")
        if streak == 3:
            return True
    print("Hết mảng → return False")
    return False

assert trace_streak([1, 2, 34, 3, 4, 5, 7, 23, 12]) is True
print()
assert trace_streak([1, 1, 2, 1, 1]) is False

arr = [1, 2, 34, 3, 4, 5, 7, 23, 12]
i=0, x= 1, lẻ=True , streak=1 
i=1, x= 2, lẻ=False, streak=0 RESET
i=2, x=34, lẻ=False, streak=0 RESET
i=3, x= 3, lẻ=True , streak=1 
i=4, x= 4, lẻ=False, streak=0 RESET
i=5, x= 5, lẻ=True , streak=1 
i=6, x= 7, lẻ=True , streak=2 
i=7, x=23, lẻ=True , streak=3 → ĐỦ 3, return True

arr = [1, 1, 2, 1, 1]
i=0, x= 1, lẻ=True , streak=1 
i=1, x= 1, lẻ=True , streak=2 
i=2, x= 2, lẻ=False, streak=0 RESET
i=3, x= 1, lẻ=True , streak=1 
i=4, x= 1, lẻ=True , streak=2 
Hết mảng → return False


Trace thứ hai cho thấy một "bẫy": có **bốn** số lẻ nhưng bị số `2` chia thành hai chuỗi dài 2 → `False`.

## 7. Các lời giải khác

### 7.1. Cách 3 — `zip` ba bản dịch của mảng

`zip(arr, arr[1:], arr[2:])` ghép song song ba list lệch nhau 1 vị trí → cho đúng các bộ ba kề nhau. `zip` dừng ở list **ngắn nhất**, nên tự lo biên.

In [13]:
arr = [10, 20, 30, 40, 50]
print(list(zip(arr, arr[1:], arr[2:])))

def three_odds_zip(arr: list[int]) -> bool:
    return any(a % 2 and b % 2 and c % 2 for a, b, c in zip(arr, arr[1:], arr[2:]))

print(three_odds_zip([2, 6, 4, 1]), three_odds_zip([1, 2, 34, 3, 4, 5, 7, 23, 12]))

[(10, 20, 30), (20, 30, 40), (30, 40, 50)]
False True


Ở đây `a % 2 and b % 2 and c % 2` dùng **truthiness**: `1` là "đúng", `0` là "sai". Biểu thức `and` trả về giá trị cuối cùng được tính (ví dụ `1` hoặc `0`), `any` coi nó như `True/False`.

### 7.2. Cách 4 — Tích ba số lẻ là lẻ

Tích của các số nguyên **lẻ khi và chỉ khi** mọi thừa số đều lẻ (chỉ cần một thừa số chẵn là tích chẵn). Vậy kiểm tra `arr[i] * arr[i+1] * arr[i+2]` lẻ là đủ.

In [14]:
def three_odds_product(arr: list[int]) -> bool:
    for i in range(len(arr) - 2):
        if (arr[i] * arr[i + 1] * arr[i + 2]) % 2 == 1:
            return True
    return False

print(3 * 5 * 7, (3 * 5 * 7) % 2, "|", 3 * 4 * 7, (3 * 4 * 7) % 2)
print(three_odds_product([2, 6, 4, 1]), three_odds_product([1, 2, 34, 3, 4, 5, 7, 23, 12]))

105 1 | 84 0
False True


Mẹo này đẹp nhưng ở ngôn ngữ có giới hạn số nguyên, phép nhân có thể tràn số. Ở đây `1000³ = 10⁹` vẫn nhỏ hơn giới hạn `int32` (~`2.1·10⁹`) nhưng đã khá sát; nếu nhân 4 số thì tràn. Python có số nguyên không giới hạn nên không lo.

### 7.3. Cách 5 — Biến thành chuỗi và tìm `"111"`

Đổi mỗi số thành ký tự `'1'` (lẻ) hoặc `'0'` (chẵn), rồi hỏi `"111" in chuỗi`. Cách "nhìn mảng như chuỗi" rất hữu ích cho các bài pattern.

In [15]:
def three_odds_string(arr: list[int]) -> bool:
    bits = "".join(str(x % 2) for x in arr)
    return "111" in bits

print("".join(str(x % 2) for x in [1, 2, 34, 3, 4, 5, 7, 23, 12]))
print(three_odds_string([2, 6, 4, 1]), three_odds_string([1, 2, 34, 3, 4, 5, 7, 23, 12]))

100101110
False True


### 7.4. Bảng so sánh

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Cửa sổ 3 | kiểm tra `arr[i..i+2]` | `O(n)` (`O(n·k)` nếu tổng quát) | `O(1)` | đúng gợi ý đề |
| 2. **Streak** | đếm lẻ liên tiếp, reset khi chẵn | **`O(n)`** với mọi `k` | `O(1)` | **nên dùng**, tổng quát tốt |
| 3. `zip` | ghép ba bản lệch | `O(n)` | `O(n)` (do slicing) | ngắn gọn |
| 4. Tích | tích lẻ ⇔ cả ba lẻ | `O(n)` | `O(1)` | mẹo toán |
| 5. Chuỗi `"111"` | mảng → chuỗi bit | `O(n)` | `O(n)` | tư duy pattern |

## 8. Code nộp LeetCode

In [16]:
class Solution:
    def threeConsecutiveOdds(self, arr: list[int]) -> bool:
        streak = 0
        for x in arr:
            if x % 2 == 1:
                streak += 1
                if streak == 3:
                    return True
            else:
                streak = 0
        return False

sol = Solution()
print(sol.threeConsecutiveOdds([2, 6, 4, 1]))
print(sol.threeConsecutiveOdds([1, 2, 34, 3, 4, 5, 7, 23, 12]))

False
True


## 9. Vì sao đúng?

**Invariant:** sau khi xử lý `arr[0..i]`, `streak` = độ dài **chuỗi số lẻ dài nhất kết thúc đúng tại `i`** (nếu `arr[i]` chẵn thì là 0) — nhưng bị "chặn" ở 3 vì ta return ngay khi đạt 3.

- `arr[i]` lẻ: chuỗi lẻ kết thúc tại `i` = chuỗi kết thúc tại `i-1` nối thêm `arr[i]` → `streak + 1` ✓.
- `arr[i]` chẵn: không có chuỗi lẻ nào kết thúc tại `i` → `0` ✓.

**Không bỏ sót:** nếu tồn tại `i` với `arr[i-2], arr[i-1], arr[i]` đều lẻ thì tại `i` ta có `streak >= 3`; vì streak tăng từng đơn vị, nó **phải đi qua đúng giá trị 3** tại vị trí sớm nhất như vậy → return `True`.

**Không trả nhầm:** `streak == 3` chỉ xảy ra khi 3 phần tử cuối đều lẻ (do mỗi lần chẵn là reset) → có bộ ba hợp lệ.

## 10. Time / Space

Gọi `n = len(arr)`.

- **Streak:** một vòng, mỗi bước `O(1)` → **Time `O(n)`**, **Space `O(1)`**. Có thể dừng sớm ngay khi tìm thấy.
- **Cửa sổ:** `n - 2` cửa sổ × 3 phép → `O(n)`; nếu tổng quát `k` thì `O(n·k)`.
- **zip / chuỗi:** `O(n)` thời gian nhưng tạo thêm list/chuỗi → `O(n)` bộ nhớ.

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.threeConsecutiveOdds([2, 6, 4, 1]), False)
check("đề 2", sol.threeConsecutiveOdds([1, 2, 34, 3, 4, 5, 7, 23, 12]), True)
check("n=1", sol.threeConsecutiveOdds([1]), False)
check("n=2 lẻ", sol.threeConsecutiveOdds([1, 3]), False)
check("n=3 đủ", sol.threeConsecutiveOdds([1, 3, 5]), True)
check("ở cuối mảng", sol.threeConsecutiveOdds([2, 2, 1, 1, 1]), True)
check("ở đầu mảng", sol.threeConsecutiveOdds([1, 1, 1, 2, 2]), True)
check("lẻ không liền", sol.threeConsecutiveOdds([1, 2, 3, 4, 5]), False)
check("hai cụm 2", sol.threeConsecutiveOdds([1, 1, 2, 1, 1]), False)
check("toàn chẵn", sol.threeConsecutiveOdds([2] * 1000), False)
check("giá trị lớn", sol.threeConsecutiveOdds([999, 1000, 999, 997, 995]), True)

PASS đề 1: False
PASS đề 2: True
PASS n=1: False
PASS n=2 lẻ: False
PASS n=3 đủ: True
PASS ở cuối mảng: True
PASS ở đầu mảng: True
PASS lẻ không liền: False
PASS hai cụm 2: False
PASS toàn chẵn: False
PASS giá trị lớn: True


**Phản ví dụ cho cách "đếm tổng số lẻ":** người mới có thể đếm tổng số số lẻ `>= 3`. Sai, vì không đảm bảo liên tiếp:

In [18]:
def three_odds_count_wrong(arr):
    return sum(x % 2 for x in arr) >= 3

for arr in [[1, 2, 3, 4, 5], [1, 1, 2, 1, 1], [1, 3, 5]]:
    print(arr, "| đếm tổng:", three_odds_count_wrong(arr), "| đúng:", three_odds_streak(arr))

[1, 2, 3, 4, 5] | đếm tổng: True | đúng: False
[1, 1, 2, 1, 1] | đếm tổng: True | đúng: False
[1, 3, 5] | đếm tổng: True | đúng: True


### 11.1. Đối chiếu 5 cách trên input ngẫu nhiên

In [19]:
import random
rng = random.Random(1550)
for _ in range(3000):
    n = rng.randint(1, 15)
    arr = [rng.choice([1, 2, 3, 4, 5, 7, 999, 1000]) for _ in range(n)]
    exp = three_odds_window(arr)
    assert three_odds_any(arr) == exp
    assert three_odds_streak(arr) == exp
    assert bool(three_odds_zip(arr)) == exp
    assert three_odds_product(arr) == exp
    assert three_odds_string(arr) == exp
    assert Solution().threeConsecutiveOdds(arr) == exp
print("3000 test ngẫu nhiên: 5 cách khớp nhau")

3000 test ngẫu nhiên: 5 cách khớp nhau


`bool(...)` để chắc chắn so sánh `True/False` (cách `zip` dùng truthiness nhưng `any` vốn đã trả bool — thêm `bool` cho an toàn khi đọc code).

## 12. Lỗi người mới hay gặp

### Lỗi 1 — `range(len(arr))` rồi đọc `arr[i+2]`

In [20]:
def three_odds_index_bug(arr):
    for i in range(len(arr)):                    # SAI biên
        if arr[i] % 2 and arr[i + 1] % 2 and arr[i + 2] % 2:
            return True
    return False

try:
    print(three_odds_index_bug([2, 6, 4, 1]))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


Ở `[2,6,4,1]`, mọi bộ đều có số chẵn ở đầu nên `and` ngắn mạch che giấu lỗi cho tới `i = 3`: `arr[3] = 1` lẻ → đọc `arr[4]` → **IndexError**. **Sửa:** `range(len(arr) - 2)`.

### Lỗi 2 — Quên reset `streak` khi gặp số chẵn

In [21]:
def three_odds_no_reset(arr):
    streak = 0
    for x in arr:
        if x % 2 == 1:
            streak += 1
            if streak == 3:
                return True
    return False

print("sai:", three_odds_no_reset([1, 2, 3, 4, 5]), "| đúng:", three_odds_streak([1, 2, 3, 4, 5]))

sai: True | đúng: False


Không reset thì `streak` thành "tổng số lẻ", giống phản ví dụ ở mục 11.

### Lỗi 3 — Nhầm "liên tiếp" là giá trị liên tiếp
Kiểm tra `arr[i+1] == arr[i] + 2` (các số lẻ liên tiếp **về giá trị** như 3,5,7) là đọc sai đề: `[5, 7, 23]` là hợp lệ dù 7 và 23 không liên tiếp về giá trị.

### Lỗi 4 — `return False` bên trong vòng lặp
```python
for i in range(len(arr) - 2):
    if ...: return True
    else: return False      # SAI: chỉ xét bộ ba đầu tiên!
```
**Sửa:** chỉ `return False` **sau** vòng lặp.

In [22]:
def three_odds_early_false(arr):
    for i in range(len(arr) - 2):
        if arr[i] % 2 and arr[i + 1] % 2 and arr[i + 2] % 2:
            return True
        else:
            return False
    return False

print("sai:", three_odds_early_false([2, 1, 1, 1]), "| đúng:", three_odds_streak([2, 1, 1, 1]))

sai: False | đúng: True


## 13. Mẹo nhận dạng pattern

- "Có `k` phần tử **liên tiếp** thỏa tính chất P không?" → **streak counter**: `cnt = cnt + 1 if P(x) else 0`.
- "Chuỗi dài nhất các phần tử liên tiếp thỏa P" → streak + `best = max(best, cnt)` (ví dụ LeetCode 485 *Max Consecutive Ones*).
- Kích thước cửa sổ **cố định nhỏ** → kiểm tra trực tiếp `arr[i..i+k-1]` hoặc `zip(arr, arr[1:], ...)`.
- Chỉ quan tâm một tính chất nhị phân (chẵn/lẻ, có/không) → có thể "mã hóa" mảng thành chuỗi `0/1` rồi tìm pattern.
- Câu hỏi "có tồn tại?" → **return sớm** khi tìm thấy.

## 14. Biến thể

### 14.1. `k` số lẻ liên tiếp

In [23]:
def k_consecutive_odds(arr, k):
    streak = 0
    for x in arr:
        streak = streak + 1 if x % 2 else 0
        if streak >= k:
            return True
    return False

print(k_consecutive_odds([1, 3, 5, 7, 2], 4), k_consecutive_odds([1, 3, 5, 7, 2], 5))

True False


### 14.2. Độ dài chuỗi số lẻ liên tiếp dài nhất

In [24]:
def longest_odd_run(arr):
    best = streak = 0
    for x in arr:
        streak = streak + 1 if x % 2 else 0
        best = max(best, streak)
    return best

print(longest_odd_run([1, 2, 34, 3, 4, 5, 7, 23, 12]))   # 3
print(longest_odd_run([2, 4]))                           # 0

3
0


### 14.3. Vị trí bắt đầu của bộ ba lẻ đầu tiên
Khi `streak` vừa đạt 3 tại `i`, bộ ba bắt đầu ở `i - 2`.

In [25]:
def first_triple_start(arr):
    streak = 0
    for i, x in enumerate(arr):
        streak = streak + 1 if x % 2 else 0
        if streak == 3:
            return i - 2
    return -1

print(first_triple_start([1, 2, 34, 3, 4, 5, 7, 23, 12]))   # 5
print(first_triple_start([2, 6, 4, 1]))                     # -1

5
-1


## 15. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
Lập bảng `streak` cho `arr = [3, 5, 8, 7, 9, 11, 4]`. Hàm trả gì và dừng ở vị trí nào?

### Bài 2 — Đoán output
```python
arr = [1, 3, 5, 6]
print(list(zip(arr, arr[1:], arr[2:])))
print("".join(str(x & 1) for x in arr))
```

### Bài 3 — TODO: viết bản streak

In [26]:
def exercise_three_odds(arr: list[int]) -> bool:
    # TODO 1: streak = 0
    # TODO 2: lẻ → streak += 1, nếu == 3 return True; chẵn → streak = 0
    # TODO 3: return False sau vòng lặp
    pass

# assert exercise_three_odds([2, 6, 4, 1]) is False
# assert exercise_three_odds([1, 2, 34, 3, 4, 5, 7, 23, 12]) is True

### Bài 4 — TODO: ba số chẵn liên tiếp

In [27]:
def exercise_three_evens(arr: list[int]) -> bool:
    # TODO
    pass

# assert exercise_three_evens([2, 4, 6]) is True
# assert exercise_three_evens([2, 4, 5, 6, 8]) is False

### Bài 5 — TODO: đếm số bộ ba lẻ liên tiếp
Đếm số vị trí `i` mà `arr[i], arr[i+1], arr[i+2]` đều lẻ (các bộ có thể chồng lên nhau). Ví dụ `[1,1,1,1]` → 2.

In [28]:
def exercise_count_triples(arr: list[int]) -> int:
    # TODO: dùng streak — mỗi khi streak >= 3 thì có thêm một bộ kết thúc tại đây
    pass

# assert exercise_count_triples([1, 1, 1, 1]) == 2
# assert exercise_count_triples([1, 2, 34, 3, 4, 5, 7, 23, 12]) == 1

### Bài 6 — Mini challenge: đổi tối thiểu
Mỗi thao tác cho phép cộng 1 vào một phần tử (đổi chẵn ↔ lẻ). Tìm số thao tác **ít nhất** để mảng có ba số lẻ liên tiếp (trả `-1` nếu `len(arr) < 3`).

In [29]:
def exercise_min_changes(arr: list[int]) -> int:
    # TODO: với mỗi cửa sổ 3, chi phí = số phần tử chẵn trong cửa sổ; lấy min
    pass

# assert exercise_min_changes([2, 6, 4, 1]) == 2
# assert exercise_min_changes([1, 3, 5]) == 0
# assert exercise_min_changes([2, 2]) == -1

## 16. Gợi ý 3 mức

**Bài 1:** (1) Lẻ +1, chẵn về 0. (2) Đi lần lượt từng số. (3) Đạt 3 ở số `11`.

**Bài 2:** (1) `zip` dừng ở list ngắn nhất. (2) `x & 1` là bit cuối. (3) Hai bộ ba; chuỗi 4 ký tự.

**Bài 3:** (1) Một biến đếm. (2) `if x % 2 == 1 ... else ...`. (3) Xem mục 5.

**Bài 4:** (1) Đổi điều kiện. (2) `x % 2 == 0`. (3) Phần còn lại y hệt.

**Bài 5:** (1) Một chuỗi lẻ dài `L` chứa `L - 2` bộ ba. (2) Không reset count khi đạt 3. (3) `if streak >= 3: total += 1`.

**Bài 6:** (1) Chỉ cần xét từng cửa sổ 3. (2) Chi phí cửa sổ = số chẵn trong đó (mỗi số chẵn +1 thành lẻ). (3) `min(sum(x % 2 == 0 for x in arr[i:i+3]) for i in ...)`.

## 17. Đáp án tham khảo

**Bài 1:**

| x | 3 | 5 | 8 | 7 | 9 | 11 |
|---|---|---|---|---|---|---|
| streak | 1 | 2 | 0 | 1 | 2 | 3 → `True` |

Dừng ở vị trí 5 (`x = 11`); số `4` không được đọc.

**Bài 2:** `[(1, 3, 5), (3, 5, 6)]` và `"1110"`.

In [30]:
def answer_three_odds(arr):
    streak = 0
    for x in arr:
        if x % 2 == 1:
            streak += 1
            if streak == 3:
                return True
        else:
            streak = 0
    return False

def answer_three_evens(arr):
    streak = 0
    for x in arr:
        streak = streak + 1 if x % 2 == 0 else 0
        if streak == 3:
            return True
    return False

def answer_count_triples(arr):
    streak = total = 0
    for x in arr:
        streak = streak + 1 if x % 2 else 0
        if streak >= 3:
            total += 1
    return total

def answer_min_changes(arr):
    if len(arr) < 3:
        return -1
    return min(sum(1 for x in arr[i:i + 3] if x % 2 == 0) for i in range(len(arr) - 2))

assert answer_three_odds([3, 5, 8, 7, 9, 11, 4]) is True
assert answer_three_odds([2, 6, 4, 1]) is False
assert answer_three_odds([1, 2, 34, 3, 4, 5, 7, 23, 12]) is True
assert answer_three_evens([2, 4, 6]) is True
assert answer_three_evens([2, 4, 5, 6, 8]) is False
assert answer_count_triples([1, 1, 1, 1]) == 2
assert answer_count_triples([1, 2, 34, 3, 4, 5, 7, 23, 12]) == 1
assert answer_min_changes([2, 6, 4, 1]) == 2
assert answer_min_changes([1, 3, 5]) == 0
assert answer_min_changes([2, 2]) == -1
arr = [1, 3, 5, 6]
assert list(zip(arr, arr[1:], arr[2:])) == [(1, 3, 5), (3, 5, 6)]
assert "".join(str(x & 1) for x in arr) == "1110"
rng = random.Random(5)
for _ in range(500):
    a = [rng.randint(1, 6) for _ in range(rng.randint(3, 12))]
    brute = sum(1 for i in range(len(a) - 2) if all(x % 2 for x in a[i:i + 3]))
    assert answer_count_triples(a) == brute
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def threeConsecutiveOdds(self, arr: list[int]) -> bool:
        streak = 0
        for x in arr:
            if x % 2 == 1:          # lẻ: nối dài chuỗi
                streak += 1
                if streak == 3:
                    return True
            else:                   # chẵn: cắt chuỗi
                streak = 0
        return False
```

**Câu thần chú:** *"Liên tiếp → streak: đúng thì +1, sai thì về 0."*

**Checklist trước khi submit**
- [ ] Reset `streak = 0` khi gặp số chẵn.
- [ ] `return False` đặt **sau** vòng lặp.
- [ ] Nếu dùng chỉ số: `range(len(arr) - 2)`.
- [ ] Test mảng ngắn (`n < 3`), bộ ba ở đầu/cuối, số lẻ bị số chẵn chia cắt.
- [ ] Đúng tên method `threeConsecutiveOdds`, trả `bool`.